# README

- **Author**: `<AUTHOR NAME>`
- **Created At**: `<YYYY-MM-DD>`
- **Last Modified At**: `<YYYY-MM-DD>`

---

## What does this file do?

- `<Describe the processing purpose, unit of observation, and selection rules>`
- `<Describe joins, recoding, aggregation, and handling of missing data as applicable>`
- `<If producing train/test files, explain the split and fit preprocessing on training data only>`

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/raw/<INPUT_FILE_NAME.csv>`
     - Input key: `<SOURCE_KEY>`
     - Description: `<What does this input contain? Include keys and relevant types>`
     - Documentation: `data/raw/<INPUT_DOCUMENTATION.ipynb-or-Rmd>`
     - SHA-256: `<expected fingerprint from the exact input inspected before generating this notebook>`
  2. `<Add each additional raw or upstream processed input and its documentation; do not use a glob>`

---

## What does this file output?

- `data/processed/<OUTPUT_FILE_NAME.csv>`
  - Output key: `<OUTPUT_KEY>`
  - Description: `<What does this output contain?>`
  - Role: `<analysis | train | test>`
  - Documentation: `data/processed/<OUTPUT_DOCUMENTATION.ipynb-or-Rmd>`
  - Expected schema and row-count rules: `<Specify the intended result and checks>`
  - Verification status / SHA-256: `<pending until saved and independently reopened>`
  - Downstream finding input: `data/processed/<OUTPUT_FILE_NAME.csv>` (same exact file)
- `<Add each additional output, with its own role, documentation, and verification>`

All paths are relative to the project root, not the operating-system root.
Replace placeholders and reader/writer options before running. Keep this list,
the code below, each processed-data document, and `PROJECT.md` Data Artifacts
consistent. Do not mark planned outputs verified. The scaffold uses CSV; adapt
both writing and read-back code together for another actual format. Bind every
input to its inspected SHA-256 before data loading; do not silently accept a
changed version or replace the expected fingerprint during a normal rerun.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


In [ ]:
# Add each input and output explicitly, keyed by a meaningful dataset name.
INPUTS = {"<SOURCE_KEY>": "data/raw/<INPUT_FILE_NAME.csv>"}
EXPECTED_INPUT_SHA256 = {"<SOURCE_KEY>": "<INSPECTED_INPUT_SHA256>"}
OUTPUTS = {"<OUTPUT_KEY>": "data/processed/<OUTPUT_FILE_NAME.csv>"}
INPUT_OPTIONS = {"<SOURCE_KEY>": {}}  # e.g. dtype, parse_dates, encoding, na_values
OUTPUT_READ_OPTIONS = {"<OUTPUT_KEY>": {}}  # Reconstruct the intended stored schema.

input_paths = {}
for key, relative_path in INPUTS.items():
    area = Path(relative_path).parts[1] if len(Path(relative_path).parts) > 1 else ""
    if area not in {"raw", "processed"}:
        raise ValueError(f"Expected a raw or upstream processed input: {relative_path}")
    input_paths[key] = project_file(relative_path, area)
output_paths = {key: project_file(path, "processed", must_exist=False) for key, path in OUTPUTS.items()}
if len(set(output_paths.values())) != len(output_paths):
    raise ValueError("Each output needs a distinct file path.")
if set(input_paths.values()) & set(output_paths.values()):
    raise ValueError("An output must not overwrite one of its inputs.")
existing = [str(path) for path in output_paths.values() if path.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")

if set(EXPECTED_INPUT_SHA256) != set(INPUTS):
    raise ValueError("Every input must have its own expected SHA-256.")
input_fingerprints = {key: verify_input_sha256(path, EXPECTED_INPUT_SHA256[key])
                      for key, path in input_paths.items()}
inputs = {key: pd.read_csv(path, **INPUT_OPTIONS.get(key, {})) for key, path in input_paths.items()}
for key, frame in inputs.items():
    print({"input": key, "data_path": INPUTS[key], "rows": len(frame),
           "schema": {column: str(dtype) for column, dtype in frame.dtypes.items()},
           "sha256": input_fingerprints[key]})


In [ ]:
# Replace this cell with the documented transformations and dataset-specific
# assertions. Keep inputs unchanged. Create one DataFrame per OUTPUTS key:
# outputs = {"<OUTPUT_KEY>": prepared_dataframe, ...}
# Check merge coverage/cardinality, exclusions, missingness, keys, and units
# as relevant. For train/test, check the declared split and leakage boundary.
raise NotImplementedError("Implement and check the actual transformations before writing any output.")


In [ ]:
# Run only after the transformation cell has produced and checked `outputs`.
assert set(outputs) == set(OUTPUTS), "Produced datasets must match the declared output paths."
for key, frame in outputs.items():
    if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
        raise ValueError(f"Output {key} must be a DataFrame with unique column names.")

verification = {}
for key, frame in outputs.items():
    path = output_paths[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    # Exclusive creation prevents an accidental overwrite during a rerun.
    with path.open("x", encoding="utf-8", newline="") as stream:
        frame.to_csv(stream, index=False)
    reread = pd.read_csv(path, **OUTPUT_READ_OPTIONS.get(key, {}))
    assert list(reread.columns) == list(frame.columns), f"Schema changed while saving {key}."
    assert len(reread) == len(frame), f"Row count changed while saving {key}."
    # Adapt storage/type normalization explicitly when needed; preserve values.
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), reread,
                                  check_dtype=False, check_categorical=False)
    verification[key] = {"data_path": OUTPUTS[key], "rows": len(reread),
                         "schema": {column: str(dtype) for column, dtype in reread.dtypes.items()},
                         "sha256": sha256_file(path), "status": "read-back checks passed"}
    print(verification[key])
assert all(sha256_file(path) == input_fingerprints[key] for key, path in input_paths.items()), "An input changed."
# Use these actual results to finish every declared processed-data document
# and Data Artifacts row. A failed write/read-back remains unverified.
# Finding notebooks must use the exact output path, matching role and SHA-256,
# never the first CSV, a raw source, or a fabricated example path.
